# 00 - Oven Analysis

Parses furnace logs, plots temperature vs time, exports plateau summary tables.
Verify the plateau before running notebook 01.

**Reads:** `{sample_id}/Raw oven*/*.txt`
**Writes:** `{sample_id}/Results/{condition}/Oven/oven_plot.pdf` · `plateau_table.csv`

## Quick links
- [Configuration](#configuration): sample_id, TABLE_INTERVAL_S — saved to session.json
- [Import](#import): libraries, paths, available conditions
- [Condition selector](#condition-selector): choose which conditions to process
- [Oven plots generation](#oven-plots-generation): plots and plateau tables per condition
- [Summary](#summary): overview of all processed conditions

## Configuration

In [ ]:
import json
from pathlib import Path

_cfg = json.loads(Path("session.json").read_text()) if Path("session.json").exists() else {}

sample_id        = input(f"Sample folder name: Enter to keep '{_cfg.get('sample_id', '?')}', or type new: ").strip() or _cfg.get("sample_id", "")
TABLE_INTERVAL_S = int(input(f"Plateau table interval s: Enter to keep {_cfg.get('TABLE_INTERVAL_S', 300)}, or type new: ").strip() or _cfg.get("TABLE_INTERVAL_S", 300))

Path("session.json").write_text(json.dumps({"sample_id": sample_id, "TABLE_INTERVAL_S": TABLE_INTERVAL_S}, indent=2))

## Import

In [ ]:
import sys
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

NOTEBOOK_DIR = Path().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from pipeline.matching import find_furnace_log, parse_oven_file, plot_oven, extract_plateau_table
import pandas as pd

sample_dir = NOTEBOOK_DIR / sample_id

# glob handles trailing space in folder name
raw_oven_candidates = list(sample_dir.glob("Raw oven*"))
if not raw_oven_candidates:
    raise FileNotFoundError(f"No 'Raw oven' folder found in {sample_dir}")
raw_oven_dir = raw_oven_candidates[0]

raw_data_dir = sample_dir / "Raw data"
all_conditions = sorted([d.name for d in raw_data_dir.iterdir() if d.is_dir()])

print(f"Sample   : {sample_id}")
print(f"Raw oven : {raw_oven_dir.name}")
print(f"{len(all_conditions)} condition(s) available")

## Condition selector

Deselect conditions to skip (Ctrl/Cmd + click). All selected = process all.
Adjust selection, then run the next cell.

In [ ]:
w_conditions = widgets.SelectMultiple(
    options=all_conditions,
    value=tuple(all_conditions),
    layout=widgets.Layout(width="560px", height=f"{min(200, 30 + 22 * len(all_conditions))}px")
)
display(w_conditions)

## Oven plots generation

In [ ]:
conditions = list(w_conditions.value)
summary_rows = []

for condition_folder in conditions:
    print(f"\n{'='*70}")
    print(f"Condition: {condition_folder}")
    print(f"{'='*70}")

    results_dir = sample_dir / "Results" / condition_folder / "Oven"
    results_dir.mkdir(parents=True, exist_ok=True)

    try:
        furnace_log_path = find_furnace_log(sample_dir, condition_folder)
        print(f"Furnace log: {furnace_log_path.name}")
    except FileNotFoundError as e:
        print(f"  [SKIP] {e}")
        continue

    parsed = parse_oven_file(furnace_log_path)
    df = parsed["df"]
    plateau_df_all = df[(df["Tsample"] >= 390) & (df["Tsample"] <= 610)]
    print(f"Start dt   : {parsed['start_dt']}")
    print(f"End dt     : {df['abs_datetime'].iloc[-1]}")
    print(f"T range    : {df['Tsample'].min():.1f} — {df['Tsample'].max():.1f} °C")
    if len(plateau_df_all) > 0:
        print(f"pO2 range  : {plateau_df_all['pO2'].min():.6f} — {plateau_df_all['pO2'].max():.6f} bar  (plateau only)")
    else:
        print(f"pO2 range  : n/a (no plateau data)")

    pdf_path = results_dir / f"oven_plot_{condition_folder}.pdf"
    plot_oven(parsed, save_path=pdf_path, show=True)

    plateau_df = extract_plateau_table(parsed, interval_s=TABLE_INTERVAL_S)
    csv_path = results_dir / f"plateau_table_{condition_folder}.csv"
    plateau_df.to_csv(csv_path, index=False)
    print(f"Plateau table: {len(plateau_df)} rows saved -> {csv_path.name}")

    # show plateau window only (T in [390, 610])
    plateau_valid = plateau_df[
        (plateau_df["Tsample (°C)"] >= 390) & (plateau_df["Tsample (°C)"] <= 610)
    ]
    print(f"\nPlateau window ({len(plateau_valid)} rows):")
    pd.set_option("display.float_format", "{:.6f}".format)
    pd.set_option("display.max_rows", 100)
    display(plateau_valid.reset_index(drop=True))

    summary_rows.append({
        "condition":   condition_folder,
        "start_dt":    str(parsed['start_dt']),
        "end_dt":      str(df['abs_datetime'].iloc[-1]),
        "T_min":       round(df['Tsample'].min(), 1),
        "T_max":       round(df['Tsample'].max(), 1),
        "pO2_min":     plateau_df_all["pO2"].min() if len(plateau_df_all) > 0 else float("nan"),
        "pO2_max":     plateau_df_all["pO2"].max() if len(plateau_df_all) > 0 else float("nan"),
        "n_points":    len(df),
        "furnace_log": furnace_log_path.name,
    })

print(f"\n{'='*70}")
print(f"Stage 0 complete: {len(summary_rows)} condition(s) processed.")

## Summary

In [ ]:
if summary_rows:
    df_summary = pd.DataFrame(summary_rows)
    print("Summary:")
    display(df_summary)

**Next step:** verify the plots and plateau tables, then run [01_ism_labeling.ipynb](01_ism_labeling.ipynb#01---ism-labeling)